# FlowGuard -- AML detection: train, evaluate, visualize (datathon VM edition)

Runs the frozen `configs/experiment.yaml` protocol (E0 rule baseline -> E1
transaction-only XGBoost -> E2 XGBoost + graph/behaviour features) end to end,
built **only** from the packages in the datathon VM's `pip list` (no pyarrow,
no shap, no numpy 2, no snapml 1.17.2 / xgboost 3.x -- the repo's own
`pyproject.toml`/`requirements.lock` target a different machine and are not
usable here as-is).

**What this notebook does differently from the WSL dev pipeline:**
- Reads the raw CSVs directly; never writes or reads Parquet (no pyarrow here).
- Reuses the repo's own `src/flowguard` modules directly (`schema`, `loader`,
  `patterns`, `windowing`, `validator`, `splits.temporal`, `features.*`,
  `models.xgb`, `models.rules`, `evaluation.*`) -- these only import
  pandas/numpy/scikit-learn, so they run unmodified.
- The Graph Feature Preprocessor (`snapml.GraphFeaturePreprocessor`) is
  attempted; if this VM's snapml build can't construct it, the notebook falls
  back to a lightweight pandas-only graph-lite feature set instead of failing.
- GFP's own chunked-extraction path writes Parquet; here it's swapped for
  `.npy` chunks (see the "E2" section) so the whole corpus can be processed
  without ever exceeding this VM's 6 GB RAM.
- Feature importance uses XGBoost's own gain importance (no shap).

**Before running:** edit the `RAW_DIR` and `FLOWGUARD_SRC` paths in the next
code cell to match where you put the dataset and the `flowguard` repo on this
VM.


In [ ]:
# --- Environment / package-version sanity check -----------------------------
# Fails loudly and early if a required package is missing or the wrong major
# version, rather than 40 minutes into the GFP extraction.
import sys, os, platform, importlib.metadata as ilm

print("python       :", sys.version.split()[0])
print("platform     :", platform.platform())
print("machine      :", platform.machine(), "/ byte order:", sys.byteorder)
print("cpu count    :", os.cpu_count())

if platform.machine() == "s390x":
    print("\n  IBM LinuxONE (s390x, big-endian) detected.")
    print("  - No PyPI wheels exist for most packages here; install nothing.")
    print("  - snapml 1.16.0's GraphFeaturePreprocessor is VERIFIED working on")
    print("    this platform (215 engineered features, ~7,900 tx/s). The E2 cell")
    print("    still probes it and falls back if that ever stops being true.")
    print("  - All artifacts this notebook writes (JSON / CSV / .npy) are")
    print("    endian-safe; no pickles are produced, deliberately.")

REQUIRED = {
    "numpy": "1.", "pandas": "2.", "scikit-learn": "1.", "xgboost": "2.",
    "scipy": "1.", "matplotlib": "3.", "seaborn": "0.", "networkx": "3.",
    "joblib": "1.", "PyYAML": "5.",
}
problems = []
for pkg, want_prefix in REQUIRED.items():
    try:
        v = ilm.version(pkg)
    except ilm.PackageNotFoundError:
        problems.append(f"  MISSING: {pkg}")
        continue
    marker = "OK" if v.startswith(want_prefix) else "UNEXPECTED VERSION"
    print(f"  {pkg:14s} {v:10s} [{marker}]")
    if marker != "OK":
        problems.append(f"  {pkg} is {v}, expected {want_prefix}x")

# snapml is optional at this stage -- the GFP cell below handles its absence.
try:
    print(f"  {'snapml':14s} {ilm.version('snapml')}")
except ilm.PackageNotFoundError:
    print("  snapml         not installed (E2 will use the graph-lite fallback)")

for name in ("pyarrow", "shap"):
    try:
        ilm.version(name)
        print(f"  WARNING: {name} is installed but this notebook does not use it "
              f"(kept off the required list on purpose)")
    except ilm.PackageNotFoundError:
        pass

if problems:
    print("\n".join(problems))
    raise RuntimeError(
        "Package check failed -- see above. This notebook is built strictly "
        "against the datathon VM's pip list; install exactly those versions."
    )
print("\nAll required packages present at compatible versions.")


## 1. Paths (auto-detected)

The next cell finds the `flowguard` source tree and the dataset by itself.
It only stops if it genuinely cannot find them, and then it prints every
location it checked so you know exactly what to fix.

Override order, highest priority first:
1. the `*_OVERRIDE` variables in the cell below,
2. the `FLOWGUARD_SRC` / `FLOWGUARD_RAW_DIR` environment variables,
3. auto-detection.

> **Do not run `pip install -e .` for this repo on this VM.** Its
> `pyproject.toml` requires Python 3.12, pyarrow, shap, numpy>=2,
> snapml 1.17.2 and xgboost>=3 -- installing it would try to change the
> provided environment. This notebook never needs it: it puts the source
> tree on `sys.path` instead, and the import cell asserts that's the copy
> actually in use.


In [ ]:
import os
import sys
from pathlib import Path

VARIANT = "HI-Small"

# ---- Optional manual overrides (leave as None to auto-detect) -------------
FLOWGUARD_SRC_OVERRIDE = None   # e.g. Path("/home/linux1/flowguard/src")
RAW_DIR_OVERRIDE = None         # e.g. Path("/home/linux1/data/IBM_Dataset")
# ---------------------------------------------------------------------------

HOME = Path.home()
CWD = Path.cwd()


def _dedupe(paths):
    seen, out = set(), []
    for p in paths:
        key = str(p)
        if key not in seen:
            seen.add(key)
            out.append(p)
    return out


def _find_flowguard_src():
    """Locate the repo's 'src' directory (the one holding flowguard/data/schema.py)."""
    if FLOWGUARD_SRC_OVERRIDE:
        return Path(FLOWGUARD_SRC_OVERRIDE), []
    if os.environ.get("FLOWGUARD_SRC"):
        return Path(os.environ["FLOWGUARD_SRC"]), []

    candidates = _dedupe([
        # notebook sitting inside the repo (linuxone/ -> ../src)
        *[parent / "src" for parent in (CWD, *CWD.parents)],
        # any directory under $HOME holding a src/, whatever it is called --
        # this is what saves us when the kernel's cwd is $HOME rather than the
        # notebook's own directory, which is exactly how a bare kernel starts.
        *sorted(HOME.glob("*/src")),
        *sorted(HOME.glob("*/*/src")),
    ])
    for c in candidates:
        if (c / "flowguard" / "data" / "schema.py").exists():
            return c, candidates
    return None, candidates


# pandas reads .csv.gz transparently, so the two CSVs may be in either form --
# which keeps ~475 MB off this VM's disk. Patterns.txt must stay uncompressed:
# parse_patterns() reads it with a plain open(), not through pandas.
CSV_SUFFIXES = (".csv", ".csv.gz")


def _resolve(directory, stem, suffixes):
    for suffix in suffixes:
        candidate = directory / f"{stem}{suffix}"
        if candidate.exists():
            return candidate
    return None


def _find_raw_dir():
    """Locate the directory holding the raw IBM AML-World CSV/TXT files."""
    if RAW_DIR_OVERRIDE:
        return Path(RAW_DIR_OVERRIDE), []
    if os.environ.get("FLOWGUARD_RAW_DIR"):
        return Path(os.environ["FLOWGUARD_RAW_DIR"]), []

    candidates = _dedupe([
        HOME / "data" / "IBM_Dataset", HOME / "data" / "ibm_dataset",
        HOME / "data", HOME / "IBM_Dataset", HOME / "Dataset_" / "IBM_Dataset",
        CWD / "data" / "IBM_Dataset", CWD / "data",
        CWD.parent / "data" / "IBM_Dataset", CWD.parent / "data",
        HOME / "datathon_research" / "Dataset_" / "IBM_Dataset",
    ])
    for c in candidates:
        if _resolve(c, f"{VARIANT}_Trans", CSV_SUFFIXES):
            return c, candidates

    # Last resort: one bounded scan of $HOME. Stops at the first hit.
    try:
        for needle in (f"{VARIANT}_Trans.csv", f"{VARIANT}_Trans.csv.gz"):
            for hit in HOME.rglob(needle):
                return hit.parent, candidates
    except Exception:
        pass
    return None, candidates


FLOWGUARD_SRC, src_tried = _find_flowguard_src()
if FLOWGUARD_SRC is None or not (FLOWGUARD_SRC / "flowguard" / "data" / "schema.py").exists():
    raise FileNotFoundError(
        "Could not find the flowguard source tree (a 'src' directory containing "
        "flowguard/data/schema.py).\n\nChecked:\n"
        + "\n".join(f"  {p}" for p in src_tried)
        + "\n\nFix: copy the repo's src/ directory onto this VM, then either set "
          "FLOWGUARD_SRC_OVERRIDE at the top of this cell or export FLOWGUARD_SRC."
    )

RAW_DIR, raw_tried = _find_raw_dir()
if RAW_DIR is None:
    raise FileNotFoundError(
        f"Could not find {VARIANT}_Trans.csv anywhere.\n\nChecked:\n"
        + "\n".join(f"  {p}" for p in raw_tried)
        + f"\n  ...and a recursive scan of {HOME}"
        + "\n\nFix: put the IBM AML-World files somewhere on this VM, then either "
          "set RAW_DIR_OVERRIDE at the top of this cell or export FLOWGUARD_RAW_DIR."
    )

TRANS_PATH = _resolve(RAW_DIR, f"{VARIANT}_Trans", CSV_SUFFIXES)
ACCOUNTS_PATH = _resolve(RAW_DIR, f"{VARIANT}_accounts", CSV_SUFFIXES)
PATTERNS_PATH = _resolve(RAW_DIR, f"{VARIANT}_Patterns", (".txt",))

missing = [label for label, path in (
    (f"{VARIANT}_Trans.csv (or .csv.gz)", TRANS_PATH),
    (f"{VARIANT}_Patterns.txt", PATTERNS_PATH),
    (f"{VARIANT}_accounts.csv (or .csv.gz)", ACCOUNTS_PATH),
) if path is None]
if missing:
    raise FileNotFoundError(
        f"Found {RAW_DIR}, but these required files are missing:\n"
        + "\n".join(f"  {m}" for m in missing)
        + "\n\nAll three are needed. Patterns.txt must be uncompressed."
    )
required_files = [TRANS_PATH, PATTERNS_PATH, ACCOUNTS_PATH]

OUTPUT_DIR = Path(os.environ.get("FLOWGUARD_OUTPUT_DIR", HOME / "flowguard_outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- How much of the corpus to run on ------------------------------------
#
# This VM has 6 GB, of which ~4.1 GB is usable after the OS, the Jupyter
# server and this kernel's imports. The pipeline costs ~2,200 B/row at peak
# (the dominant term is pandas storing account ids as Python str objects --
# 61 B each, not 4), so the full 5.08M-row corpus needs ~11.2 GB and will be
# OOM-killed here. The reference machine measured 10.99 GB on exactly this
# run, which is what that estimate is calibrated against.
#
# WINDOW_DAYS selects whole days instead of a row prefix, because a prefix is
# a poor sample of this corpus: 2022/09/01 alone is 1.11M rows but only 322
# positives (0.029%), so `nrows` spends the entire memory budget on the most
# positive-poor stretch. Measured per-day counts:
#
#     09/01  1,114,921 rows    322 pos  0.029%      09/06  482,089   531  0.110%
#     09/02    754,449 rows    408 pos  0.054%      09/07  482,751   497  0.103%
#     09/03    207,382 rows    391 pos  0.189%      09/08  482,773   539  0.112%
#     09/04    207,430 rows    407 pos  0.196%      09/09  654,467   514  0.079%
#     09/05    482,650 rows    471 pos  0.098%      09/10  208,325   442  0.212%
#
# MEASURED ON THE VM, not estimated.
#
# 09/08-09/09 -- 1,137,240 rows, 1,053 positives, 127 in test -- runs end to
# end in 6.3 minutes at a 2.50 GB peak, and is the default. E2 reaches PR-AUC
# 0.0311 there against E1's 0.0198.
#
# Feature extraction scales far past this: the pipeline has built the HDF5 for
# the FULL 5,077,237-row corpus on this machine -- a 4.05 GB file, 189
# features, GFP over every edge at 4,058 tx/s, 3.79 GB peak. What does not fit
# is the final XGBoost fit: QuantileDMatrix over 3.55M x 189 exceeds the
# remaining budget, and with no swap the box thrashes instead of failing
# cleanly. Training the whole corpus needs external-memory XGBoost (DMatrix
# with cache_prefix, spilling to the 32 GB of free disk) or more RAM.
#
# So widen this freely for feature extraction; re-measure before training.
WINDOW_DAYS = ("2022/09/08", "2022/09/09")   # inclusive, YYYY/MM/DD

# Additional hard cap on rows, applied after windowing. None = no cap.
SAMPLE_ROWS = None

sys.path.insert(0, str(FLOWGUARD_SRC))
print("FLOWGUARD_SRC:", FLOWGUARD_SRC)
print("RAW_DIR      :", RAW_DIR)
print("OUTPUT_DIR   :", OUTPUT_DIR)
print("SAMPLE_ROWS  :", SAMPLE_ROWS or "(full dataset)")
print("\nraw files:")
for p in required_files:
    print(f"  {p.name:26s} {p.stat().st_size / 1e6:>10,.1f} MB")


In [ ]:
# --- Imports -----------------------------------------------------------
# Everything below (schema/loader/patterns/windowing/validator/splits/
# features.transaction/features.behaviour/models.xgb/models.rules/
# evaluation.*) is pandas/numpy/scikit-learn only -- no pyarrow, no shap,
# no snapml at import time. Verified by reading every import line in the
# repo before writing this notebook.
import gc
import json
import time
import resource
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

import flowguard

# Guard: make sure we are running the source tree found above, not a
# pip-installed copy shadowing it. A `pip install -e .` of this repo would
# drag in pyarrow/shap/numpy>=2/snapml 1.17.2/xgboost>=3 and break this VM's
# environment -- if that ever happened, fail here rather than halfway through.
_loaded_from = Path(flowguard.__file__ or "").resolve().parent
if not str(_loaded_from).startswith(str(Path(FLOWGUARD_SRC).resolve())):
    raise RuntimeError(
        f"'flowguard' imported from {_loaded_from}, not from the expected "
        f"source tree {FLOWGUARD_SRC}. A pip-installed copy is shadowing it. "
        "Run `pip uninstall flowguard` (do NOT `pip install -e .` on this VM) "
        "and restart the kernel."
    )

from flowguard.data import schema as S
from flowguard.data.loader import load_transactions, load_accounts
from flowguard.data.patterns import parse_patterns, attach_patterns
from flowguard.data.windowing import trim_sparse_tail, daily_profile
from flowguard.data.validator import validate_transactions
from flowguard.splits.temporal import SplitSpec, chronological_split
from flowguard.features.transaction import TransactionFeatures
from flowguard.features.behaviour import behaviour_features
from flowguard.features.gfp import GFPFeatures, windowed_params
from flowguard.models.xgb import XGBModel, DEFAULT_PARAMS
from flowguard.models.rules import RuleBaseline
from flowguard.evaluation.metrics import evaluate, per_group_recall, DEFAULT_BUDGETS
from flowguard.evaluation.sanity import run_null_baselines, shuffled_label_test

assert hasattr(xgb, "QuantileDMatrix"), (
    "xgboost.QuantileDMatrix is missing -- XGBModel.fit() requires it "
    "(xgboost >= 2.0). Check `import xgboost; xgboost.__version__`."
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100


def peak_rss_gb() -> float:
    """Peak resident set size of this process so far, in GB (Linux, stdlib only)."""
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6


def checkpoint(label: str) -> None:
    gc.collect()
    print(f"[{label}] peak RSS so far: {peak_rss_gb():.2f} GB", flush=True)


checkpoint("imports done")


## 2. Ingest

Loads the raw CSVs straight into the canonical schema, attaches laundering
pattern labels, trims the generator's sparse tail, and validates -- exactly
the steps `flowguard.pipeline.ingest` runs, minus the Parquet write (which
needs pyarrow) and minus the SHA-256/provenance bookkeeping (not needed for
training).


In [ ]:
started = time.perf_counter()

from flowguard.data.loader import RAW_TRANSACTION_COLUMNS


def window_by_day(source, destination, first_day, last_day, chunk=250_000):
    """Copy only the rows whose day falls in [first_day, last_day] to a new CSV.

    Streamed in chunks and filtered on the raw timestamp *string*, so the full
    corpus is never resident -- which is the whole point, since materialising
    it is exactly what this VM cannot afford. The filtered file is then read by
    the ordinary load_transactions(), so the subset goes through precisely the
    same parsing, typing and validation as a full run.
    """
    kept = total = 0
    header = pd.read_csv(source, nrows=0)
    with open(destination, "w", newline="", encoding="utf-8") as out:
        out.write(",".join(header.columns) + "\n")
        for part in pd.read_csv(source, skiprows=1, header=None,
                                names=list(RAW_TRANSACTION_COLUMNS),
                                dtype=str, chunksize=chunk):
            total += len(part)
            day = part["timestamp_raw"].str.slice(0, 10)
            selected = part[(day >= first_day) & (day <= last_day)]
            kept += len(selected)
            selected.to_csv(out, header=False, index=False)
            print(f"      scanned {total:>10,}  kept {kept:>10,}", end="\r", flush=True)
    print()
    return kept, total


source_path = TRANS_PATH
if WINDOW_DAYS:
    windowed = OUTPUT_DIR / f"{VARIANT}_Trans_{WINDOW_DAYS[0].replace('/', '')}_{WINDOW_DAYS[1].replace('/', '')}.csv"
    if windowed.exists():
        print(f"[0/4] reusing windowed corpus {windowed.name}")
    else:
        print(f"[0/4] windowing {TRANS_PATH.name} to {WINDOW_DAYS[0]}..{WINDOW_DAYS[1]} ...")
        kept, total = window_by_day(TRANS_PATH, windowed, *WINDOW_DAYS)
        print(f"      kept {kept:,} of {total:,} rows ({kept / total:.1%})")
    source_path = windowed

print(f"[1/4] loading {source_path.name} ...")
if source_path.suffix == ".gz":
    print("      (gzipped -- pandas decompresses on the fly; expect it to be slower)")
tx, load_report = load_transactions(source_path, amount_side="paid", nrows=SAMPLE_ROWS)
print(f"      {len(tx):,} transactions, {int(tx[S.IS_LAUNDERING].sum()):,} positives")

print(f"[2/4] parsing {PATTERNS_PATH.name} ...")
patterns = parse_patterns(PATTERNS_PATH)
tx, attach_report = attach_patterns(tx, patterns)
_r = attach_report
coverage = (_r.labelled_positives / _r.total_positives) if _r.total_positives else 0.0
print(f"      {_r.patterns} patterns, {_r.matched_transactions:,} transactions "
      f"labelled ({coverage:.1%} of positives)")

print("[3/4] trimming the generator's sparse tail ...")
tx, window_report = trim_sparse_tail(tx, min_density=0.05)
if window_report.applied:
    print(f"      dropped {window_report.rows_dropped:,} rows "
          f"({window_report.positives_dropped:,} positives) across "
          f"{len(window_report.days_dropped)} trailing day(s)")
else:
    print("      no sparse tail detected")

print("[4/4] validating ...")
validation = validate_transactions(tx)
print(validation.summary())
if not validation.ok:
    raise RuntimeError("Validation found ERROR-severity issues -- fix the data before training.")

summary = S.summarise(tx)
print(f"\nfinal corpus: {summary.rows:,} rows, {summary.accounts:,} accounts, "
      f"{summary.positives:,} positives, base rate {summary.base_rate:.4%}")
print(f"span: {summary.ts_min} .. {summary.ts_max}")
print(f"ingest took {time.perf_counter() - started:.0f}s")
checkpoint("ingest done")


## 2b. Slim the transaction frame

pandas has no pyarrow here, so every string column is stored as Python
objects: one account id costs **61 bytes**, not 4. Eight such columns make the
frame 547 B/row (measured), which at 5.08M rows is 2.8 GB before a single
feature exists.

Encoding accounts as int32 codes and the low-cardinality columns as
categoricals takes it to roughly 150 B/row. That is the difference between the
full corpus fitting in this VM's 3.9 GB and not. Nothing downstream needs the
strings: GFP wants dense integer vertex ids anyway, and the account labels are
kept in `ACCOUNT_LABELS` for the few nodes the graph view draws.

This runs **after** pattern attachment, which needs the original strings to
build its natural join key.


In [ ]:
SLIM_DTYPES = True
ACCOUNT_LABELS = None

if SLIM_DTYPES:
    before_bytes = tx.memory_usage(deep=True).sum()

    codes, ACCOUNT_LABELS = pd.factorize(
        pd.concat([tx[S.SOURCE_ACCOUNT], tx[S.DESTINATION_ACCOUNT]], ignore_index=True),
        sort=False)
    n = len(tx)
    tx[S.SOURCE_ACCOUNT] = codes[:n].astype("int32")
    tx[S.DESTINATION_ACCOUNT] = codes[n:].astype("int32")
    del codes
    gc.collect()

    # currency and currency_received are compared elementwise by the rule
    # baseline and the transaction features. Two categoricals can only be
    # compared when they share a category set, so build one dtype for both.
    shared = pd.CategoricalDtype(
        sorted(set(tx[S.CURRENCY].dropna()) | set(tx[S.CURRENCY_RECEIVED].dropna())))
    tx[S.CURRENCY] = tx[S.CURRENCY].astype(shared)
    tx[S.CURRENCY_RECEIVED] = tx[S.CURRENCY_RECEIVED].astype(shared)
    for col in (S.PAYMENT_TYPE, S.PATTERN_TYPE, S.SCENARIO_ID):
        tx[col] = tx[col].astype("category")

    # transaction_id is TX{position:010d} -- the position itself is the id.
    tx[S.TRANSACTION_ID] = np.arange(n, dtype="int32")
    tx[S.AMOUNT] = tx[S.AMOUNT].astype("float32")
    tx[S.AMOUNT_RECEIVED] = tx[S.AMOUNT_RECEIVED].astype("float32")
    gc.collect()

    after_bytes = tx.memory_usage(deep=True).sum()
    print(f"tx frame: {before_bytes/1e6:,.0f} MB -> {after_bytes/1e6:,.0f} MB "
          f"({before_bytes/after_bytes:.1f}x smaller, "
          f"{before_bytes/n:.0f} -> {after_bytes/n:.0f} B/row)")
    print(f"  {len(ACCOUNT_LABELS):,} distinct accounts encoded as int32")
    S.validate_schema(tx)          # the contract still holds
checkpoint("dtypes slimmed")


## 3. Chronological split

`SplitSpec()` defaults (70% train / 15% val / 15% test, `HARD_CUT` boundary
policy, seed 42) match the frozen `configs/experiment.yaml` exactly -- see
`docs/ADR-002-boundary-policy.md` for why `HARD_CUT` was chosen over `PURGE`.


In [ ]:
split = chronological_split(tx, SplitSpec(seed=42))
train_df, val_df, test_df = split.apply(tx)

print(f"train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")
print(f"boundaries: train_end={split.train_end}  val_end={split.val_end}")
print(f"test positives: {int(test_df[S.IS_LAUNDERING].sum()):,}")
print(f"patterns spanning a boundary: {split.patterns_spanning_boundary}")
for note in split.notes:
    print(f"note: {note}")

y_train = train_df[S.IS_LAUNDERING].to_numpy()
y_val = val_df[S.IS_LAUNDERING].to_numpy()
y_test = test_df[S.IS_LAUNDERING].to_numpy()

assert len(train_df) + len(val_df) + len(test_df) == len(tx), "split dropped or duplicated rows"
assert y_test.sum() > 0, "test partition has zero positives -- split boundaries are broken"


## 4. E0 -- rule baseline

Fixed thresholds, fitted on the training partition only. Exists to be beaten.


In [ ]:
results = {}

rules = RuleBaseline().fit(train_df)
e0_val = evaluate(y_val, rules.score(val_df))
e0_test = evaluate(y_test, rules.score(test_df))
print("E0 (rule baseline) -- test:")
print(e0_test.summary())
results["E0"] = {"val": e0_val.to_metadata(), "test": e0_test.to_metadata()}


## 5. Sanity checks (leakage guard)

Two null baselines (random / constant scorers) and a shuffled-label test.
If the shuffled-label run scores meaningfully above the test base rate,
label information is leaking through a path other than the label itself,
and every metric after this cell is void. Uses a fast (100-tree) model on
transaction-only features -- same code path as E1, just cheaper.


In [ ]:
tx_extractor_probe = TransactionFeatures().fit(S.feature_view(train_df))
X_train_probe = tx_extractor_probe.run(S.feature_view(train_df))
X_test_probe = tx_extractor_probe.run(S.feature_view(test_df))

sanity_results = run_null_baselines(y_test, seed=42)


def _fit_predict(xt, yt, xs):
    m = XGBModel()
    m.n_estimators = 100
    m.fit(xt, yt)
    return m.predict_raw(xs)


shuffled = shuffled_label_test(_fit_predict, X_train_probe, y_train, X_test_probe, y_test, seed=42)
sanity_results.append(shuffled)

all_passed = True
for r in sanity_results:
    status = "PASS" if r.passed else "FAIL"
    print(f"  [{status}] {r.name:16s} PR-AUC={r.metrics.pr_auc:.6f}   {r.criterion}")
    all_passed &= r.passed

results["sanity_passed"] = all_passed
del X_train_probe, X_test_probe, tx_extractor_probe
checkpoint("sanity checks done")

assert all_passed, (
    "SANITY FAILURE -- a null or shuffled-label baseline scored above its "
    "ceiling. The pipeline is leaking; do not trust E1/E2 below until this "
    "is fixed."
)


## 6. E1 -- transaction-only XGBoost

Row-local features only (amount, time-of-day, currency, payment rail). No
cross-row information. This is the floor E2's graph/behaviour features must
clear.


In [ ]:
tx_extractor = TransactionFeatures().fit(S.feature_view(train_df))
X_train_tx = tx_extractor.run(S.feature_view(train_df))
X_val_tx = tx_extractor.run(S.feature_view(val_df))
X_test_tx = tx_extractor.run(S.feature_view(test_df))
print(f"{X_train_tx.shape[1]} transaction features: {list(X_train_tx.columns)}")

e1_model = XGBModel(n_estimators=300)
e1_model.fit(X_train_tx, y_train, X_val_tx, y_val)
e1_model.calibrate(X_val_tx, y_val)

e1_val = evaluate(y_val, e1_model.predict(X_val_tx))
e1_test = evaluate(y_test, e1_model.predict(X_test_tx))
print("\nE1 (transaction-only) -- test:")
print(e1_test.summary())
print(f"trained in {e1_model.train_seconds_:.1f}s on device={e1_model.resolved_device_}, "
      f"best_iteration={e1_model.best_iteration_}")

e1_importance = e1_model.importance(top=15)
print("\ntop features by gain:")
total_gain = sum(e1_importance.values()) or 1.0
for name, gain in e1_importance.items():
    print(f"  {name:28s} {gain / total_gain:6.1%}")

results["E1"] = {
    "val": e1_val.to_metadata(), "test": e1_test.to_metadata(),
    "importance_gain": e1_importance, "model_meta": e1_model.to_metadata(),
}
checkpoint("E1 done")


## 7. E2 -- graph (GFP) + behaviour features

Two feature families join the transaction-only ones from E1:

1. **Behaviour features** (`flowguard.features.behaviour`): causal
   account-history counts/amounts, computed once over the whole
   chronologically ordered corpus with an O(n log n) two-pointer scan --
   pure numpy/pandas, no risk on this VM.
2. **GFP features** (`flowguard.features.gfp`, wrapping
   `snapml.GraphFeaturePreprocessor`): fan-in/out, degree, scatter-gather,
   temporal-cycle and vertex-statistics features from IBM's C++ graph engine.
   This is the highest-risk step on this VM -- the repo has only verified
   `snapml==1.17.2` on WSL2/Linux (`docs/ADR-001`); this VM has `1.16.0`.
   GFP is Linux-only (not Windows), and this VM *is* Linux, so it has a real
   chance of working -- but it is untested at this exact version.

   **The cell below tries GFP first. If construction fails for any reason,
   it automatically falls back to a lightweight pandas-only "graph-lite"
   feature set** (in/out degree and distinct-counterparty counts in a
   trailing window, computed the same causal way as the behaviour features)
   so the notebook still produces a real E2 result instead of crashing.
   Which path ran is recorded in `results["E2"]["feature_source"]`.

**Memory note:** GFP's own chunked-extraction path writes Parquet part files
(pyarrow, unavailable here). The subclass below overrides just the two I/O
methods to use `.npy` chunks instead, so the graph itself still streams
through the whole 5M-row corpus one edge at a time (bounded memory) without
ever calling pyarrow.


In [ ]:
GFP_CACHE = OUTPUT_DIR / f"{VARIANT}_gfp_npy_cache"


class NpyChunkedGFP(GFPFeatures):
    """GFPFeatures, but flushes chunks as .npy instead of .parquet.

    Only I/O changes; the graph-insertion logic (run_streaming) is untouched
    -- reused verbatim from the repo. This exists solely because pyarrow
    (needed for GFPFeatures' built-in Parquet chunking) is not in this VM's
    package list.
    """

    def run_streaming(self, *args, **kwargs):
        # The parent clears stale chunks with glob("part_*.parquet"), which
        # never matches the .npy parts written here. Without this, a shorter
        # second run reads its own parts PLUS the longer previous run's, and
        # the row count silently disagrees with the corpus.
        if self.chunk_dir is not None and self.chunk_dir.exists():
            for stale in self.chunk_dir.glob("part_*.npy"):
                stale.unlink()
        return super().run_streaming(*args, **kwargs)

    def _flush_chunk(self, block: np.ndarray, index: pd.Index) -> None:
        if self.n_engineered_ is None:
            self.n_engineered_ = block.shape[1]
        stem = self.chunk_dir / f"part_{self.n_parts_written_:05d}"
        np.save(f"{stem}_data.npy", block)
        np.save(f"{stem}_index.npy", index.to_numpy())
        self.n_parts_written_ += 1


def _npy_parts(chunk_dir: Path):
    return sorted(chunk_dir.glob("part_*_data.npy"))


def npy_varying_columns(chunk_dir: Path) -> np.ndarray:
    """Boolean mask of GFP output columns that are not constant corpus-wide.

    Mirrors flowguard.features.gfp.varying_columns, but reads .npy parts
    (memory-mapped, so this pass never holds more than one part in RAM).
    """
    lo = hi = None
    for path in _npy_parts(chunk_dir):
        block = np.load(path, mmap_mode="r")
        cmin, cmax = np.nanmin(block, axis=0), np.nanmax(block, axis=0)
        lo = cmin if lo is None else np.minimum(lo, cmin)
        hi = cmax if hi is None else np.maximum(hi, cmax)
    return hi > lo


def read_npy_varying_chunks(chunk_dir: Path, order: pd.Index) -> pd.DataFrame:
    """Assemble the GFP feature frame from .npy parts, varying columns only.

    Mirrors flowguard.features.gfp.read_varying_chunks.
    """
    keep = npy_varying_columns(chunk_dir)
    kept_idx = np.flatnonzero(keep)
    print(f"  {len(kept_idx)} of {len(keep)} GFP features vary; reading only those", flush=True)

    blocks, idxs = [], []
    for path in _npy_parts(chunk_dir):
        block = np.load(path)[:, keep]
        idx = np.load(str(path).replace("_data.npy", "_index.npy"))
        blocks.append(block)
        idxs.append(idx)
    data = np.concatenate(blocks, axis=0)
    row_index = np.concatenate(idxs, axis=0)
    del blocks, idxs
    gc.collect()

    columns = [f"gfp_f{i:03d}" for i in kept_idx]
    frame = pd.DataFrame(data, index=pd.Index(row_index, name="_row"), columns=columns)
    del data
    gc.collect()
    return frame.reindex(order)


def graph_lite_features(tx_view: pd.DataFrame) -> pd.DataFrame:
    """Fallback graph-ish features if snapml's GFP is unavailable here.

    Pure pandas/numpy, causal (strictly-earlier events only, like
    behaviour_features). NOT the pre-registered E2 feature set -- this is a
    degraded substitute so the notebook still produces a real (if weaker)
    E2 result instead of stopping.
    """
    from flowguard.features.behaviour import _Timeline, DAY

    S.assert_label_blind(tx_view)
    ts = tx_view[S.TIMESTAMP]
    t = ((ts - ts.min()).dt.total_seconds()).to_numpy(dtype="int64")
    amount = tx_view[S.AMOUNT].to_numpy(dtype="float64")

    accounts, _ = pd.factorize(
        pd.concat([tx_view[S.SOURCE_ACCOUNT], tx_view[S.DESTINATION_ACCOUNT]]), sort=False
    )
    n = len(tx_view)
    src, dst = accounts[:n].astype("int64"), accounts[n:].astype("int64")

    sent = _Timeline(src, t, amount)
    received = _Timeline(dst, t, amount)

    out = pd.DataFrame(index=tx_view.index)
    # out-degree / in-degree proxy: transaction counts in a trailing 2-day window
    out["glite_src_outdeg_2d"] = sent.count(src, t, 2 * DAY)
    out["glite_src_indeg_2d"] = received.count(src, t, 2 * DAY)
    out["glite_dst_outdeg_2d"] = sent.count(dst, t, 2 * DAY)
    out["glite_dst_indeg_2d"] = received.count(dst, t, 2 * DAY)
    out["glite_src_out_amt_2d"] = sent.total(src, t, 2 * DAY)
    out["glite_dst_in_amt_2d"] = received.total(dst, t, 2 * DAY)
    return out


# Availability probe ONLY. The fallback must trigger when snapml genuinely
# cannot provide GFP on this platform -- never because of a bug in the
# extraction or assembly below. A defect that silently swaps in weaker
# features and still reports a number is far worse than a crash: it was
# caught exactly once here, when stale .npy parts from a longer previous run
# made the assembled block disagree with the corpus.
GFP_AVAILABLE, gfp_unavailable_reason = True, None
try:
    from snapml import GraphFeaturePreprocessor
    GraphFeaturePreprocessor()          # construction is the real test
except Exception as exc:
    GFP_AVAILABLE = False
    gfp_unavailable_reason = f"{type(exc).__name__}: {exc}"

if GFP_AVAILABLE:
    print("snapml GraphFeaturePreprocessor available -- extracting", flush=True)
    params = windowed_params(2.0)  # matches configs/experiment.yaml features.gfp
    params["num_threads"] = max(1, os.cpu_count() or 1)  # this VM has 2 vCPUs, not 8

    gfp_extractor = NpyChunkedGFP(
        params=params,
        batch_size=1,              # MUST stay 1 -- see gfp.py docstring (ADR re: double-insertion)
        exclude_self_transfers=True,
        chunk_dir=GFP_CACHE,
        chunk_rows=200_000,        # ~180 MB/part at float32 x ~230 cols
        progress_every=500_000,
    )
    cached = sorted(GFP_CACHE.glob("part_*_index.npy"))
    cached_rows = sum(len(np.load(c, mmap_mode="r")) for c in cached) if cached else 0
    if cached_rows == len(tx):
        # Extraction is the long pole (~18 min on the full corpus). Reuse is
        # only safe when the cache covers exactly this many rows -- a mismatch
        # is how stale parts silently corrupted a run once already.
        print(f"  reusing {len(cached)} cached GFP parts ({cached_rows:,} rows)", flush=True)
    else:
        if cached_rows:
            print(f"  cache holds {cached_rows:,} rows for a {len(tx):,}-row corpus "
                  f"-- re-extracting", flush=True)
        gfp_extractor.run_streaming(S.feature_view(tx), assemble=False)
        print(f"  GFP extraction: {gfp_extractor.extract_seconds_:.0f}s, "
              f"{gfp_extractor.n_edges_inserted_ / (gfp_extractor.extract_seconds_ or 1):,.0f} tx/s",
              flush=True)
    # Anything below this line failing is a defect -- let it raise.
    # Only the column mask is computed here. Assembling the block would cost
    # 156 x 4 bytes per row for the whole corpus purely to hand it straight
    # to the writer; the streaming cell reads the parts from disk instead.
    GFP_KEEP_MASK = npy_varying_columns(GFP_CACHE)
    gfp_feats = None
    GFP_FEATURE_SOURCE = "gfp"
    print(f"  GFP: {int(GFP_KEEP_MASK.sum())} of {len(GFP_KEEP_MASK)} features vary "
          f"(streamed from disk, never assembled)", flush=True)
else:
    warnings.warn(
        f"snapml GraphFeaturePreprocessor is unavailable here "
        f"({gfp_unavailable_reason}). Falling back to graph-lite features. "
        f"This E2 result will NOT match the pre-registered GFP feature set.",
        stacklevel=2,
    )
    gfp_feats = graph_lite_features(S.feature_view(tx))
    GFP_KEEP_MASK = None
    GFP_FEATURE_SOURCE = "graph_lite_fallback"
    print(f"  graph-lite fallback: {gfp_feats.shape[1]} features", flush=True)
    assert len(gfp_feats) == len(tx) and gfp_feats.index.equals(tx.index), (
        "graph-lite block is misaligned with the corpus"
    )

checkpoint("graph features done")


In [ ]:
started = time.perf_counter()
bh_feats = behaviour_features(S.feature_view(tx))
print(f"behaviour features in {time.perf_counter() - started:.0f}s: "
      f"{bh_feats.shape[1]} features")
checkpoint("behaviour features done")


In [ ]:
# Stream the E2 feature matrix straight into HDF5, one chunk at a time.
#
# The previous version built it in pandas -- graph block, three partition
# matrices and a concat copy all resident at once, ~3,150 B/row. That is what
# OOM-killed the kernel at 1.14M rows. Here the matrix is never materialised:
# each GFP chunk is read, joined to its transaction and behaviour features,
# written, and dropped. Peak becomes one chunk, so RAM stops scaling with the
# corpus and the full 5.08M rows fit.
#
# Rows go in GFP's chronological order, which -- because the split is a hard
# chronological cut -- makes train, val and test three contiguous ranges. No
# shuffling is needed and a partition is one slice.
import h5py

H5_PATH = OUTPUT_DIR / f"{VARIANT}_features.h5"
CHUNK = 200_000

order = tx[S.TIMESTAMP].sort_values(kind="stable").index   # exactly what GFP used
sorted_ts = tx[S.TIMESTAMP].loc[order]
N_TRAIN = int((sorted_ts <= split.train_end).sum())
N_VAL = int(((sorted_ts > split.train_end) & (sorted_ts <= split.val_end)).sum())
N_TEST = len(tx) - N_TRAIN - N_VAL
y_ordered = tx[S.IS_LAUNDERING].loc[order].to_numpy().astype("int8")
assert (N_TRAIN, N_VAL, N_TEST) == tuple(split.sizes.values()), \
    f"chronological ranges {N_TRAIN, N_VAL, N_TEST} disagree with the split {split.sizes}"

bh_cols = list(bh_feats.columns)
bh_ordered = bh_feats.loc[order].to_numpy(dtype="float32")
del bh_feats
gc.collect()

if GFP_FEATURE_SOURCE == "gfp":
    keep_mask = GFP_KEEP_MASK
    gfp_cols = [f"gfp_f{i:03d}" for i in np.flatnonzero(keep_mask)]
else:
    gfp_cols = list(gfp_feats.columns)

tx_cols = list(X_train_tx.columns)
FEATURE_NAMES = tx_cols + bh_cols + gfp_cols
n_features = len(FEATURE_NAMES)
print(f"streaming {len(tx):,} x {n_features} into {H5_PATH.name}", flush=True)

with h5py.File(H5_PATH, "w") as h5:
    X = h5.create_dataset("X", shape=(len(tx), n_features), dtype="float32",
                          chunks=(min(8192, len(tx)), n_features))
    h5.create_dataset("y", data=y_ordered)
    h5.create_dataset("feature_names", data=np.array(FEATURE_NAMES, dtype=object),
                      dtype=h5py.string_dtype())
    # Categorical metadata goes in as int codes plus a small category table.
    # Materialising 5M Python strings per column costs ~330 MB each and, for a
    # categorical holding pd.NA, h5py refuses the conversion outright.
    for name, col in (("pattern_type", S.PATTERN_TYPE), ("scenario_id", S.SCENARIO_ID)):
        series = tx[col].loc[order]
        cat = series if isinstance(series.dtype, pd.CategoricalDtype) else series.astype("category")
        h5.create_dataset(f"{name}_codes", data=cat.cat.codes.to_numpy().astype("int32"))
        h5.create_dataset(f"{name}_categories",
                          data=cat.cat.categories.astype(str).to_numpy(),
                          dtype=h5py.string_dtype())
    ids = tx[S.TRANSACTION_ID].loc[order]
    if pd.api.types.is_integer_dtype(ids):
        h5.create_dataset("transaction_id", data=ids.to_numpy(dtype="int64"))
    else:
        h5.create_dataset("transaction_id", data=ids.astype(str).to_numpy(),
                          dtype=h5py.string_dtype())
    for name, col in (("source_account", S.SOURCE_ACCOUNT),
                      ("destination_account", S.DESTINATION_ACCOUNT)):
        column = tx[col].loc[order]
        if SLIM_DTYPES:     # int32 codes; ACCOUNT_LABELS maps them back
            h5.create_dataset(name, data=column.to_numpy(dtype="int32"))
        else:
            h5.create_dataset(name, data=column.astype(str).to_numpy(),
                              dtype=h5py.string_dtype())
    h5.create_dataset("timestamp",
                      data=sorted_ts.astype("int64").to_numpy() // 1_000_000_000)
    h5.create_dataset("amount", data=tx[S.AMOUNT].loc[order].to_numpy(dtype="float64"))

    written = 0
    if GFP_FEATURE_SOURCE == "gfp":
        sources = sorted(GFP_CACHE.glob("part_*_data.npy"))
    else:
        sources = [None]                       # fallback block is already in RAM
    for part in sources:
        if part is None:
            graph_block = gfp_feats.loc[order].to_numpy(dtype="float32")
            rows = order
        else:
            graph_block = np.load(part)[:, keep_mask]
            rows = pd.Index(np.load(str(part).replace("_data.npy", "_index.npy")))
        for lo in range(0, len(rows), CHUNK):
            hi = min(lo + CHUNK, len(rows))
            idx = rows[lo:hi]
            txf = tx_extractor.run(S.feature_view(tx.loc[idx])).to_numpy(dtype="float32")
            X[written:written + (hi - lo)] = np.hstack(
                [txf, bh_ordered[written:written + (hi - lo)], graph_block[lo:hi]])
            written += hi - lo
            print(f"    {written:>10,}/{len(tx):,}", end="\r", flush=True)
        del graph_block
        gc.collect()
    print()
    assert written == len(tx), f"wrote {written:,} rows for {len(tx):,} transactions"

    h5.attrs.update({
        "variant": VARIANT, "n_train": N_TRAIN, "n_val": N_VAL, "n_test": N_TEST,
        "n_features": n_features, "gfp_feature_source": GFP_FEATURE_SOURCE,
        "row_order": "chronological; train, then val, then test (contiguous)",
        "train_end": str(split.train_end), "val_end": str(split.val_end),
        "created": pd.Timestamp.now(tz="UTC").isoformat(),
        "nan_policy": "preserved; XGBoost uses them natively, impute before Keras/torch",
    })

TRAIN_SPAN, VAL_SPAN, TEST_SPAN = (0, N_TRAIN), (N_TRAIN, N_TRAIN + N_VAL), \
                                  (N_TRAIN + N_VAL, len(tx))
y_train_h5 = y_ordered[slice(*TRAIN_SPAN)]
y_val_h5 = y_ordered[slice(*VAL_SPAN)]
y_test_h5 = y_ordered[slice(*TEST_SPAN)]
del bh_ordered
gc.collect()
print(f"wrote {H5_PATH.name} ({H5_PATH.stat().st_size/1e9:.2f} GB)")
checkpoint("HDF5 assembled")


In [ ]:
# Train E2 straight from the HDF5 through a DataIter, so XGBoost never sees
# a materialised matrix either -- QuantileDMatrix bins each chunk as it
# arrives and keeps only the histogram (~1 byte per value).
class H5Iter(xgb.DataIter):
    """Feeds XGBoost one HDF5 slice at a time."""

    def __init__(self, path, span, names, chunk=CHUNK):
        self._path, self._lo, self._hi, self._chunk = str(path), span[0], span[1], chunk
        self._names = names
        self._i = 0
        super().__init__()

    def next(self, input_data):
        start = self._lo + self._i * self._chunk
        if start >= self._hi:
            return 0
        stop = min(start + self._chunk, self._hi)
        with h5py.File(self._path, "r") as f:
            # xgboost 2.0 rejects feature_names on a QuantileDMatrix built from
            # an iterator -- every piece of info has to arrive per batch.
            input_data(data=f["X"][start:stop], label=f["y"][start:stop],
                       feature_names=self._names)
        self._i += 1
        return 1

    def reset(self):
        self._i = 0


def predict_h5(booster, span, chunk=50_000):
    booster.set_param({"device": "cpu"})
    out = []
    with h5py.File(H5_PATH, "r") as f:
        for lo in range(span[0], span[1], chunk):
            out.append(booster.inplace_predict(f["X"][lo:min(lo + chunk, span[1])]))
    return np.concatenate(out)


# Everything the E1 pandas path held is dead weight from here on, and at full
# corpus scale it is the difference between fitting and being OOM-killed.
for _name in ("X_train_tx", "X_val_tx", "X_test_tx", "gfp_feats", "train_df", "val_df"):
    if _name in dir():
        del globals()[_name]
gc.collect()
checkpoint("freed E1 working set")

params = dict(DEFAULT_PARAMS)
params["device"] = "cpu"
params["n_jobs"] = max(1, os.cpu_count() or 1)
# 128 bins rather than the default 256: halves the sketch and the binned
# matrix, and on a 0.089% base rate the extra split resolution buys nothing
# measurable. This is a memory decision, and it is why the full corpus fits.
params["max_bin"] = 128
positives = int(y_train_h5.sum())
params["scale_pos_weight"] = (len(y_train_h5) - positives) / max(positives, 1)

started = time.perf_counter()
# Smaller batches than the writer used -- each one is materialised as float32
# before binning, so 50k x 189 is 38 MB in flight instead of 151 MB.
ITER_CHUNK = 50_000
dtrain = xgb.QuantileDMatrix(H5Iter(H5_PATH, TRAIN_SPAN, FEATURE_NAMES, ITER_CHUNK),
                             max_bin=params["max_bin"])
gc.collect()
dval = xgb.QuantileDMatrix(H5Iter(H5_PATH, VAL_SPAN, FEATURE_NAMES, ITER_CHUNK),
                           ref=dtrain, max_bin=params["max_bin"])
gc.collect()
checkpoint("QuantileDMatrix built")
# Only val is evaluated: scoring train each round costs a full extra
# prediction cache and buys nothing, since early stopping watches val.
booster = xgb.train(params, dtrain, num_boost_round=300, early_stopping_rounds=100,
                    evals=[(dval, "val")], verbose_eval=False)
train_seconds = time.perf_counter() - started
del dtrain, dval
gc.collect()
print(f"E2 trained in {train_seconds:.0f}s, best_iteration={booster.best_iteration}")
checkpoint("E2 trained")

# Isotonic calibration on validation only -- same treatment XGBModel gives.
from sklearn.isotonic import IsotonicRegression

raw_val, raw_test = predict_h5(booster, VAL_SPAN), predict_h5(booster, TEST_SPAN)
calibrator = IsotonicRegression(out_of_bounds="clip").fit(raw_val, y_val_h5)
e2_val_scores, e2_scores = calibrator.predict(raw_val), calibrator.predict(raw_test)

e2_val = evaluate(y_val_h5, e2_val_scores)
e2_test = evaluate(y_test_h5, e2_scores)
print("\nE2 (graph + behaviour) -- test:")
print(e2_test.summary())

e2_model = booster        # the charts and the save cell refer to this name

# E0/E1 score rows in the frame's own order; E2 scores them in chronological
# (HDF5) order. Keeping both straight matters -- pairing one model's scores
# with the other's labels would silently produce a plausible, wrong number.
# Everything E2-side below uses y_test_h5 and this metadata frame.
with h5py.File(H5_PATH, "r") as f:
    def _decoded(name):
        """Rebuild a categorical column from its codes and category table."""
        cats = np.array([c.decode() if isinstance(c, bytes) else str(c)
                         for c in f[f"{name}_categories"][:]], dtype=object)
        codes = f[f"{name}_codes"][slice(*TEST_SPAN)]
        out = np.full(len(codes), None, dtype=object)
        seen = codes >= 0
        out[seen] = cats[codes[seen]]
        return out

    test_meta = pd.DataFrame({
        S.TRANSACTION_ID: f["transaction_id"][slice(*TEST_SPAN)],
        S.PATTERN_TYPE: _decoded("pattern_type"),
        S.SCENARIO_ID: _decoded("scenario_id"),
        S.SOURCE_ACCOUNT: (f["source_account"][slice(*TEST_SPAN)] if SLIM_DTYPES
                           else _s("source_account")),
        S.DESTINATION_ACCOUNT: (f["destination_account"][slice(*TEST_SPAN)] if SLIM_DTYPES
                                else _s("destination_account")),
        S.AMOUNT: f["amount"][slice(*TEST_SPAN)],
        S.IS_LAUNDERING: y_test_h5,
    })

gain = booster.get_score(importance_type="gain")
e2_importance = dict(sorted(gain.items(), key=lambda kv: -kv[1])[:20])
typology_recall = per_group_recall(
    y_test_h5, e2_scores, test_meta[S.PATTERN_TYPE].to_numpy(), budget=0.01)

results["E2"] = {
    "val": e2_val.to_metadata(), "test": e2_test.to_metadata(),
    "importance_gain": e2_importance, "feature_source": GFP_FEATURE_SOURCE,
    "typology_recall_at_1pct": typology_recall,
    "model": {"n_features": len(FEATURE_NAMES), "best_iteration": booster.best_iteration,
              "scale_pos_weight": params["scale_pos_weight"],
              "train_seconds": round(train_seconds, 1),
              "trained_from": "HDF5 via QuantileDMatrix(DataIter)"},
}
checkpoint("E2 done")


## 8. Results comparison

`docs/ADR-013`/the pre-registered gates in `configs/experiment.yaml` define
"beats" as delta PR-AUC > 2 sigma (sigma measured at 0.0017 across 5 seeds on
the dev machine -- this notebook runs one seed, so treat deltas here as
indicative, not gate-passing evidence).


In [ ]:
comparison = pd.DataFrame(
    {
        "PR-AUC": [results[e]["test"]["pr_auc"] for e in ("E0", "E1", "E2")],
        "ROC-AUC": [results[e]["test"]["roc_auc"] for e in ("E0", "E1", "E2")],
        "lift_over_base_rate": [results[e]["test"]["lift_over_base_rate"] for e in ("E0", "E1", "E2")],
        "recall@1%": [
            next(b["recall"] for b in results[e]["test"]["budgets"] if b["budget"] == 0.01)
            for e in ("E0", "E1", "E2")
        ],
        "precision@1%": [
            next(b["precision"] for b in results[e]["test"]["budgets"] if b["budget"] == 0.01)
            for e in ("E0", "E1", "E2")
        ],
    },
    index=["E0 (rules)", "E1 (transaction)", f"E2 ({GFP_FEATURE_SOURCE})"],
)
print(comparison.round(4).to_string())

print(f"\nP2 gate (E1 beats E0): delta PR-AUC = "
      f"{results['E1']['test']['pr_auc'] - results['E0']['test']['pr_auc']:+.4f}")
print(f"P3 gate (E2 beats E1): delta PR-AUC = "
      f"{results['E2']['test']['pr_auc'] - results['E1']['test']['pr_auc']:+.4f}")


## 9. Visualizations

In [ ]:
# Daily volume / laundering-rate profile -- confirms the sparse tail was
# trimmed and shows the base-rate texture across the corpus window.
profile = daily_profile(tx)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax1.bar(profile.index.astype(str), profile["rows"], color="#4C72B0")
ax1.set_ylabel("transactions / day")
ax1.set_title(f"{VARIANT}: daily volume and laundering rate (post-trim)")
ax2.plot(profile.index.astype(str), profile["rate"] * 100, marker="o", color="#C44E52")
ax2.set_ylabel("laundering rate (%)")
ax2.set_xlabel("date")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "daily_profile.png")
plt.show()


In [ ]:
# Precision-recall curves, E0 vs E1 vs E2
from sklearn.metrics import precision_recall_curve, roc_curve

fig, ax = plt.subplots(figsize=(7, 6))
for name, labels, scores, color in (
    ("E0 (rules)", y_test, rules.score(test_df), "#8172B2"),
    ("E1 (transaction)", y_test, e1_model.predict(X_test_tx), "#4C72B0"),
    (f"E2 ({GFP_FEATURE_SOURCE})", y_test_h5, e2_scores, "#C44E52"),
):
    precision, recall, _ = precision_recall_curve(labels, scores)
    pr_auc = results[name.split()[0]]["test"]["pr_auc"]
    ax.plot(recall, precision, label=f"{name} (PR-AUC={pr_auc:.4f})", color=color)
base_rate = results["E0"]["test"]["base_rate"]
ax.axhline(base_rate, linestyle="--", color="gray", linewidth=1, label=f"base rate ({base_rate:.4%})")
ax.set_xlabel("recall")
ax.set_ylabel("precision")
ax.set_title(f"{VARIANT} test set -- precision/recall")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pr_curves.png")
plt.show()


In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(7, 6))
for name, labels, scores, color in (
    ("E0 (rules)", y_test, rules.score(test_df), "#8172B2"),
    ("E1 (transaction)", y_test, e1_model.predict(X_test_tx), "#4C72B0"),
    (f"E2 ({GFP_FEATURE_SOURCE})", y_test_h5, e2_scores, "#C44E52"),
):
    fpr, tpr, _ = roc_curve(labels, scores)
    roc_auc = results[name.split()[0]]["test"]["roc_auc"]
    ax.plot(fpr, tpr, label=f"{name} (ROC-AUC={roc_auc:.4f})", color=color)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
ax.set_xlabel("false positive rate")
ax.set_ylabel("true positive rate")
ax.set_title(f"{VARIANT} test set -- ROC")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curves.png")
plt.show()


In [ ]:
# Score distribution for the best model (E2), laundering vs legitimate
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(x=e2_scores[y_test_h5 == 0], stat="density", bins=60, color="#4C72B0",
             label="legitimate", alpha=0.6, ax=ax)
sns.histplot(x=e2_scores[y_test_h5 == 1], stat="density", bins=60, color="#C44E52",
             label="laundering", alpha=0.6, ax=ax)
ax.set_xlabel("E2 calibrated score")
ax.set_title(f"E2 score distribution by class ({VARIANT} test set)")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "score_distribution.png")
plt.show()


In [ ]:
# Recall / precision @ alert budget, all three experiments
budget_rows = []
for exp in ("E0", "E1", "E2"):
    for b in results[exp]["test"]["budgets"]:
        budget_rows.append({"experiment": exp, "budget": f"{b['budget']:.1%}",
                              "recall": b["recall"], "precision": b["precision"]})
budget_df = pd.DataFrame(budget_rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=budget_df, x="budget", y="recall", hue="experiment", ax=axes[0])
axes[0].set_title("Recall @ alert budget")
sns.barplot(data=budget_df, x="budget", y="precision", hue="experiment", ax=axes[1])
axes[1].set_title("Precision @ alert budget")
axes[1].set_yscale("log")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "budget_recall_precision.png")
plt.show()


In [ ]:
# Confusion matrices at two thresholds.
#
# Read these with care: at a 0.089% base rate the true-negative cell swamps
# everything, which is exactly why this project reports PR-AUC and recall@budget
# instead of accuracy (a model that never fires scores 99.9%). So colour here is
# the fraction WITHIN each true class, while the annotation is the raw count.
from sklearn.metrics import confusion_matrix

val_threshold = e2_val.best_f1_threshold           # chosen on VALIDATION, never test
budget_threshold = e2_test.at_budget(0.01).threshold

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
panels = [
    (f"best-F1 threshold from validation\n(score >= {val_threshold:.4f})", val_threshold),
    (f"1% alert budget\n(top {int(round(len(y_test) * 0.01)):,} scores)", budget_threshold),
]
for ax, (title, thr) in zip(axes, panels):
    cm = confusion_matrix(y_test, (e2_scores >= thr).astype(int), labels=[0, 1])
    cm_frac = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    sns.heatmap(cm_frac, annot=cm, fmt=",d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["predicted clean", "predicted laundering"],
                yticklabels=["actually clean", "actually laundering"])
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f"{title}\nrecall {tp / max(tp + fn, 1):.1%}  |  "
                 f"precision {tp / max(tp + fp, 1):.2%}  |  {fp:,} false alerts",
                 fontsize=10)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrices.png")
plt.show()


In [ ]:
# E2 feature importance (native XGBoost gain -- no shap on this VM)
imp = pd.Series(e2_importance).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, max(4, 0.3 * len(imp))))
imp.plot.barh(ax=ax, color="#55A868")
ax.set_xlabel("gain (relative)")
ax.set_title("E2 -- top features by XGBoost gain")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "e2_feature_importance.png")
plt.show()


In [ ]:
# Recall per laundering typology @1% budget (E2)
if typology_recall:
    typ_df = pd.DataFrame(typology_recall).T.sort_values("recall")
    fig, ax = plt.subplots(figsize=(8, max(3, 0.4 * len(typ_df))))
    ax.barh(typ_df.index, typ_df["recall"], color="#DD8452")
    ax.set_xlabel("recall @ 1% budget")
    ax.set_title("E2 -- recall by laundering typology")
    for y, (idx, row) in enumerate(typ_df.iterrows()):
        ax.text(row["recall"] + 0.01, y, f"{int(row['caught'])}/{int(row['positives'])}", va="center")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "typology_recall.png")
    plt.show()
    zero_recall = typ_df[typ_df["recall"] == 0]
    if len(zero_recall):
        print(f"P5 gate WARNING: {len(zero_recall)} typology(ies) at zero recall: "
              f"{list(zero_recall.index)}")
else:
    print("No typology labels matched this sample (pattern-attach coverage too low) -- skipping.")


In [ ]:
# The graph view: laundering patterns the model actually caught, drawn as the
# account networks they are. This is the qualitative counterpart to the metrics
# -- the fan-out / scatter-gather / cycle shapes GFP's features exist to detect.
import networkx as nx

k = max(1, int(round(len(y_test_h5) * 0.01)))        # 1% alert budget
top_idx = np.argpartition(-e2_scores, k - 1)[:k]
flagged = np.zeros(len(y_test_h5), dtype=bool)
flagged[top_idx] = True

caught = test_meta.assign(_flagged=flagged)
caught = caught[(caught[S.IS_LAUNDERING] == 1) & caught["_flagged"]
                & caught[S.SCENARIO_ID].notna()]

# Best-caught scenario per typology, keeping only ones small enough to read.
picks = []
for typology, group in caught.groupby(S.PATTERN_TYPE):
    scenario_id = group[S.SCENARIO_ID].value_counts().idxmax()
    edges = tx[tx[S.SCENARIO_ID] == scenario_id]
    if 2 <= len(edges) <= 60:
        picks.append((typology, scenario_id, edges, len(group)))
picks.sort(key=lambda p: -p[3])
picks = picks[:3]

if not picks:
    print("No caught, drawable pattern in the top 1% -- skipping the graph view.")
else:
    fig, axes = plt.subplots(1, len(picks), figsize=(6.2 * len(picks), 5.5))
    axes = np.atleast_1d(axes)
    for ax, (typology, scenario_id, edges, n_caught) in zip(axes, picks):
        G = nx.DiGraph()
        for _, row in edges.iterrows():
            G.add_edge(row[S.SOURCE_ACCOUNT], row[S.DESTINATION_ACCOUNT],
                       amount=float(row[S.AMOUNT]))
        pos = nx.spring_layout(G, seed=42, k=0.9)
        nx.draw_networkx_nodes(
            G, pos, ax=ax, node_color="#C44E52", alpha=0.85,
            node_size=[200 + 120 * G.degree(n) for n in G.nodes])
        nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#555555", width=1.4,
                               arrowsize=12, connectionstyle="arc3,rad=0.08")
        def _label(node):
            name = str(ACCOUNT_LABELS[node]) if ACCOUNT_LABELS is not None else str(node)
            return name.split(":")[-1][-5:]
        nx.draw_networkx_labels(G, pos, ax=ax, font_size=7,
                                labels={n: _label(n) for n in G.nodes})
        ax.set_title(f"{typology}   ({scenario_id})\n"
                     f"{G.number_of_nodes()} accounts, {len(edges)} transfers  --  "
                     f"{n_caught} caught @1% budget", fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "caught_patterns_graph.png")
    plt.show()
    print("Node labels are the last 5 characters of each account id. Repeated "
          "transfers between the same pair collapse to one arrow.")


## 10. Save artifacts

Models are saved via XGBoost's own JSON format (`save_model`), not pickle --
a pickle written under one numpy/xgboost version routinely fails to load
under another (see the project's own package-constraint notes). Everything
else is JSON/CSV/`.npy`, all version-independent.


In [ ]:
e1_model.booster_.save_model(str(OUTPUT_DIR / "e1_model.json"))
e2_model.save_model(str(OUTPUT_DIR / "e2_model.json"))   # e2_model is a raw Booster

(OUTPUT_DIR / "results.json").write_text(json.dumps(results, indent=2, default=str), encoding="utf-8")
comparison.to_csv(OUTPUT_DIR / "comparison.csv")

np.save(OUTPUT_DIR / "e2_test_scores.npy", e2_scores)
np.save(OUTPUT_DIR / "test_labels.npy", y_test)
test_df[[S.TRANSACTION_ID, S.PATTERN_TYPE]].assign(e2_score=e2_scores).to_csv(
    OUTPUT_DIR / "e2_test_predictions.csv", index=False
)

print("wrote:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" ", p.name)


## 11. Final self-check

A broken run should fail here, not silently produce numbers. This is the
notebook's equivalent of a `demo()`/`assert` smoke test.


In [ ]:
assert results["sanity_passed"], "sanity checks failed -- see section 5"
assert results["E1"]["test"]["pr_auc"] > results["E0"]["test"]["base_rate"], (
    "E1 did not beat the base rate -- something upstream is broken"
)
assert not np.isnan(results["E2"]["test"]["pr_auc"]), "E2 PR-AUC is NaN"
assert len(FEATURE_NAMES) > X_train_tx.shape[1], (
    "E2's feature matrix is not larger than E1's -- the graph/behaviour join failed"
)
assert len(e2_scores) == len(y_test_h5), "E2 scores and labels have different lengths"

print("All self-checks passed.")
print(f"\nfinal peak RSS: {peak_rss_gb():.2f} GB (VM budget: 6 GB)")
print(f"\nE0 PR-AUC={results['E0']['test']['pr_auc']:.4f}  "
      f"E1 PR-AUC={results['E1']['test']['pr_auc']:.4f}  "
      f"E2 PR-AUC={results['E2']['test']['pr_auc']:.4f}  "
      f"(E2 feature source: {GFP_FEATURE_SOURCE})")


## 12. Verify the HDF5

The feature file was written during E2 rather than afterwards -- it *is* the
training input, not an export of it. This checks it round-trips before
`02` depends on it.


In [ ]:
with h5py.File(H5_PATH, "r") as h5:
    print("datasets:", ", ".join(h5.keys()))
    for key, value in h5.attrs.items():
        print(f"  {key:20s} {value}")
    assert h5["X"].shape == (len(tx), len(FEATURE_NAMES)), "X shape mismatch"
    assert int(h5["y"][:].sum()) == int(tx[S.IS_LAUNDERING].sum()),         "positive count changed on write"
    assert h5.attrs["n_train"] + h5.attrs["n_val"] + h5.attrs["n_test"] == len(tx)
    block = h5["X"][slice(*TEST_SPAN)][:512]
    assert np.isfinite(block).any(), "test block is entirely non-finite"

print(f"\nHDF5 verified: {H5_PATH.stat().st_size/1e9:.2f} GB, "
      f"{len(FEATURE_NAMES)} features. 02_keras_model.ipynb can train from it.")
